In [ ]:
# Install Library

!pip install transformers torch scikit-learn

In [ ]:
# Upload Dataset

from google.colab import files
uploaded = files.upload()

In [ ]:
# Load Dataset

import pandas as pd

df = pd.read_csv("IMDB Dataset.csv")
df.head()

In [ ]:
# Explore Data

df.info()
df['sentiment'].value_counts()

In [ ]:
# Preprocessing

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['review'] = df['review'].apply(clean_text)

In [ ]:
df['label'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [ ]:
df = df.sample(200)

In [ ]:
# Data Splitting

from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['review'], df['label'], test_size=0.2, random_state=42
)

print("Split done")

In [ ]:
# Tokenization

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(list(train_texts), truncation=True, padding=True)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True)

print("Tokenization done")

In [ ]:
# Dataset Class

import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

print("Dataset ready")

In [ ]:
# Load BERT Model

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Model loaded")

In [ ]:
# Metrics

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:
# Training Arguments

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    report_to="none"
)


In [ ]:
# Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer ready")

In [ ]:
# Train Model

trainer.train()

In [ ]:
# Evaluate Model

result = trainer.evaluate()
print(result)

In [ ]:
# Freeze BERT Layers

from transformers import AutoModelForSequenceClassification

model_frozen = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
for param in model_frozen.bert.parameters():
    param.requires_grad = False

In [ ]:
trainer_frozen = Trainer(
    model=model_frozen,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer_frozen.train()

In [ ]:
result_frozen = trainer_frozen.evaluate()
print(result_frozen)

In [ ]:
# Fine-tuned Last 2 Layers

model_last2 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
for param in model_last2.bert.parameters():
    param.requires_grad = False

In [ ]:
for param in model_last2.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

In [ ]:
trainer_last2 = Trainer(
    model=model_last2,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer_last2.train()

In [ ]:
result_last2 = trainer_last2.evaluate()
print(result_last2)

In [ ]:
# Compare Result

import pandas as pd

results = pd.DataFrame({
    "Experiment": ["Normal BERT", "Frozen BERT", "Last 2 Layers"],
    "Accuracy": [
        result['eval_accuracy'],
        result_frozen['eval_accuracy'],
        result_last2['eval_accuracy']
    ]
})

print(results)

**Observations:**

1. The fully trained BERT model gave the highest accuracy.
2. Freezing BERT layers reduced training time but lowered accuracy.
3. Fine-tuning the last 2 layers improved performance compared to frozen BERT.
4. There is a trade-off between training time and model performance.

**Conclusion:**

In this project, BERT was used for sentiment analysis on the IMDB dataset.
Different experiments were performed including freezing layers and partial fine-tuning.

The results show that:
- Fully fine-tuned BERT provides the best accuracy.
- Freezing layers reduces computation but affects performance.
- Fine-tuning a few layers gives a good balance.

Thus, BERT is highly effective for text classification tasks.